In [2]:
%matplotlib qt
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
import seaborn as sns
sns.set_style('whitegrid')


# Set this to the directory containing the CSV files to process.
input_dir = Path('../data/allie/tracking')

output_dir = Path('../data/allie/tracking_cleaned')

# Select bones to include in each cleaned output.
bones_to_use = [
    'Head', 'LShoulder', 'LUArm', 'Spine2', 'LHand', 'RHand',
    'RShoulder', 'RUArm', 'WaistLFront', 'WaistLBack',
    'WaistRBack', 'WaistRFront',
]
reference_bones = [
    'RShoulder'
]


def clean_csv(input_path, output_path, bones, reference_bones):
    with input_path.open('r') as f:
        lines = f.readlines()

    type_row_idx = next(
        (i for i, line in enumerate(lines) if ',Type,Bone,Bone' in line),
        None,
    )
    data_type_row_idx = next(
        (i for i, line in enumerate(lines) if line.startswith(',,Rotation,Rotation,')),
        None,
    )
    data_header_row_idx = next(
        (
            i
            for i, line in enumerate(lines)
            if line.startswith('Frame,Time (Seconds),X,Y,Z,W,' )
        ),
        None,
    )

    if None in (type_row_idx, data_type_row_idx, data_header_row_idx):
        raise ValueError(f'Could not find the expected headers in {input_path.name}')

    bone_names_full = lines[type_row_idx + 1].strip().split(',')[2:]
    data_types = lines[data_type_row_idx].strip().split(',')[2:]
    components = lines[data_header_row_idx].strip().split(',')[2:]

    col_names = ['Frame', 'Time']
    for i, data_type in enumerate(data_types):
        bone = bone_names_full[i] if i < len(bone_names_full) else f'Unknown_{i}'
        component = components[i] if i < len(components) else '?'
        bone_clean = bone.replace(':', '_').replace(' ', '_')
        col_names.append(f'{bone_clean}_{data_type}_{component}')

    data_raw = pd.read_csv(
        input_path,
        skiprows=data_header_row_idx + 1,
        header=None,
    )
    if len(data_raw.columns) != len(col_names):
        raise ValueError(
            f'{input_path.name}: expected {len(col_names)} columns, '
            f'but found {len(data_raw.columns)}'
        )
    data_raw.columns = col_names

    selected_cols = ['Frame', 'Time']
    for bone in bones:
        selected_cols.extend(
            col for col in data_raw.columns if f'{bone}_Position_' in col
        )

    data = data_raw[selected_cols].copy()
    new_names = {}
    for col in data.columns:
        if col in ('Frame', 'Time'):
            continue
        parts = col.split('_')
        bone = parts[2]
        component = parts[4]
        new_names[col] = f'{bone}_{component}'
    data.rename(columns=new_names, inplace=True)

    missing_by_body_part = {}
    for bone in bones:
        position_columns = [
            f'{bone}_{axis}'
            for axis in ('X', 'Y', 'Z')
            if f'{bone}_{axis}' in data.columns
        ]
        missing_by_body_part[bone] = int(
            data[position_columns].isna().any(axis=1).sum()
        ) if position_columns else 0

    data_unfiltered = data.copy()
    data_filtered = data.copy()
    # for col in data_filtered.columns:
    #     if col not in ('Frame', 'Time') and len(data_filtered[col]) >= 11:
    #         data_filtered[col] = savgol_filter(
    #             data_filtered[col].values,
    #             window_length=11,
    #             polyorder=1,
    #             mode='mirror',
    #         )

    missing_reference_bones = [bone for bone in reference_bones if bone not in bones]
    if missing_reference_bones:
        raise ValueError(
            f'Reference bones must be included in bones: {missing_reference_bones}'
        )

    reference_columns = [
        f'{bone}_{axis}'
        for bone in reference_bones
        for axis in ('X', 'Y', 'Z')
    ]
    missing_reference_columns = [
        column for column in reference_columns if column not in data_filtered.columns
    ]
    if missing_reference_columns:
        raise ValueError(
            f'{input_path.name}: missing reference bone columns '
            f'{missing_reference_columns}'
        )

    reference_position = pd.DataFrame({
        axis: data_filtered[
            [f'{bone}_{axis}' for bone in reference_bones]
        ].mean(axis=1)
        for axis in ('X', 'Y', 'Z')
    })

    # Express every marker relative to the multi-point torso and pelvis reference.
    for bone in bones:
        for axis in ('X', 'Y', 'Z'):
            position_column = f'{bone}_{axis}'
            if position_column in data_filtered.columns:
                data_filtered[position_column] = (
                    data_filtered[position_column] - reference_position[axis]
                )

    data = data_filtered
    dt = data['Time'].diff().mean()
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError(f'{input_path.name}: could not determine a valid time step')

    def derivative(values):
        result = np.zeros_like(values)
        result[1:-1] = (values[2:] - values[:-2]) / (2 * dt)
        result[0] = (values[1] - values[0]) / dt
        result[-1] = (values[-1] - values[-2]) / dt
        return result

    derived_columns = {}
    for bone in bones:
        for axis in ('X', 'Y', 'Z'):
            pos_col = f'{bone}_{axis}'
            if pos_col in data.columns:
                velocity = derivative(data[pos_col].values)
                derived_columns[f'{bone}_V_{axis}'] = velocity
                derived_columns[f'{bone}_A_{axis}'] = derivative(velocity)

        velocity_columns = [f'{bone}_V_{axis}' for axis in ('X', 'Y', 'Z')]
        if all(col in derived_columns for col in velocity_columns):
            derived_columns[f'{bone}_Speed'] = np.sqrt(
                sum(derived_columns[col] ** 2 for col in velocity_columns)
            )

    data = pd.concat(
        [data, pd.DataFrame(derived_columns, index=data.index)],
        axis=1,
    )

    data.to_csv(output_path, index=False)
    return data, data_unfiltered, missing_by_body_part



cleaned_data = {}
missing_frames_by_file = {}
for csv_path in sorted(input_dir.glob('*.csv')):
    if csv_path.stem.endswith('_cleaned'):
        continue
    output_path = output_dir / f'{csv_path.stem}_cleaned.csv'
    cleaned_data[csv_path.name], data_unfiltered, missing_frames_by_file[csv_path.name] = clean_csv(
        csv_path,
        output_path,
        bones_to_use,
        reference_bones,
    )
    print(f'Cleaned {csv_path.name} -> {output_path.name}')

if not cleaned_data:
    raise FileNotFoundError(f'No input CSV files found in {input_dir.resolve()}')

missing_frame_records = [
    {
        'File': file_name,
        'BodyPart': body_part,
        'MissingFrames': missing_count,
    }
    for file_name, body_parts in missing_frames_by_file.items()
    for body_part, missing_count in body_parts.items()
    if missing_count > 0
 ]
missing_frame_summary = pd.DataFrame(
    missing_frame_records,
    columns=['File', 'BodyPart', 'MissingFrames'],
 )
print('Missing frames by file and body part:')
if missing_frame_summary.empty:
    print('No missing body-part frames found.')
else:
    display(missing_frame_summary.sort_values(['File', 'MissingFrames', 'BodyPart']))

# Keep the existing downstream cells working with the last processed file.
data = list(cleaned_data.values())[-1]
data_filtered = data.copy()
print(f'Processed {len(cleaned_data)} CSV file(s).')

Cleaned allie_T001_triangle_vs_circle.csv -> allie_T001_triangle_vs_circle_cleaned.csv
Cleaned allie_T002_square_vs_circle.csv -> allie_T002_square_vs_circle_cleaned.csv
Cleaned allie_T003_cross_vs_triangle.csv -> allie_T003_cross_vs_triangle_cleaned.csv
Cleaned allie_T004_heart_vs_triangle.csv -> allie_T004_heart_vs_triangle_cleaned.csv
Cleaned allie_T005_heart_vs_hexagon.csv -> allie_T005_heart_vs_hexagon_cleaned.csv
Cleaned allie_T006_diamond_vs_square.csv -> allie_T006_diamond_vs_square_cleaned.csv
Cleaned allie_T007_hexagon_vs_diamond.csv -> allie_T007_hexagon_vs_diamond_cleaned.csv
Cleaned allie_T008_hexagon_vs_triangle.csv -> allie_T008_hexagon_vs_triangle_cleaned.csv
Cleaned allie_T009_cross_vs_diamond.csv -> allie_T009_cross_vs_diamond_cleaned.csv
Cleaned allie_T010_circle_vs_heart.csv -> allie_T010_circle_vs_heart_cleaned.csv
Cleaned allie_T011_heart_vs_star.csv -> allie_T011_heart_vs_star_cleaned.csv
Cleaned allie_T012_square_vs_heart.csv -> allie_T012_square_vs_heart_cleane

In [6]:
# Plot right arm speed of 1000
tracking_1kHz = pd.read_csv("..\\data\\allie\\tracking_cleaned\\allie_T016_hexagon_vs_heart_cleaned.csv")

In [7]:
# Plot right hand XYZ trjectories for 1kHz
fig, ax = plt.subplots(4, 1, figsize=(15, 5), sharex=True)
ax[0].plot(tracking_1kHz['Time'], tracking_1kHz['RHand_X'], label='X', color='r')
ax[0].set_ylabel('X Position (mm)')
ax[1].plot(tracking_1kHz['Time'], tracking_1kHz['RHand_Y'], label='Y', color='g')
ax[1].set_ylabel('Y Position (mm)')
ax[2].plot(tracking_1kHz['Time'], tracking_1kHz['RHand_Z'], label='Z', color='b')
ax[2].set_ylabel('Z Position (mm)')
ax[3].plot(tracking_1kHz['Time'], tracking_1kHz['RHand_Speed'], label='Speed', color='m')
ax[3].set_ylabel('Speed (mm/s)')
ax[0].set_title('Right Hand Position')  
fig.tight_layout()
fig.show()

In [8]:
# Draw left hand and right hand XYZ trajectories on 3D plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot(tracking_1kHz['LHand_X'], tracking_1kHz['LHand_Z'], tracking_1kHz['LHand_Y'], label='Left Hand')
ax.plot(tracking_1kHz['RHand_X'], tracking_1kHz['RHand_Z'], tracking_1kHz['RHand_Y'], label='Right Hand')
ax.set_xlabel('X')
ax.set_ylabel('Z')
ax.set_zlabel('Y')
ax.legend()
fig.tight_layout()
fig.show()